# V4 Universal Football Model — Footballdata.io Ingestion

This notebook tests the new Footballdata.io API, replacing the deprecated Sofascore integration.
We will use this notebook to explore the JSON structure and build the parsing logic before wiring it into the live V4 backend.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load the API key from .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FOOTBALLDATA_API_KEY")

if not API_KEY:
    print("⚠️ API Key not found! Please check your .env file.")
else:
    print(f"✅ API Key loaded: {API_KEY[:5]}...{API_KEY[-5:]}")

## 1. Basic API Fetch Function
Let's define a helper function to hit the Footballdata.io endpoints based on their documentation.

In [ ]:
# The correct base URL according to https://footballdata.io/documentation/endpoints/
BASE_URL = "https://footballdata.io/api/v1"

def fetch_footballdata(endpoint, params=None):
    """Helper to fetch data from footballdata.io"""
    if params is None:
        params = {}
        
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        if 'response' in locals() and response is not None:
            print(f"Response text: {response.text}")
        return None

## 2. Dynamic Fixture Parsing
The Footballdata `/fixtures/today` endpoint returns `data` as a dictionary (grouped by league or date) rather than a flat list. We will safely unpack it.

In [ ]:
print("Fetching /fixtures/today...")
today_data = fetch_footballdata("fixtures/today")
match_id_to_test = None

if today_data and today_data.get("success") and "data" in today_data:
    data_payload = today_data["data"]
    
    if isinstance(data_payload, list) and len(data_payload) > 0:
        match_id_to_test = data_payload[0].get("match_id")
    elif isinstance(data_payload, dict):
        # Iterate through the keys (which might be league IDs or dates) to find the first match array
        for key, val in data_payload.items():
            if isinstance(val, list) and len(val) > 0:
                match_id_to_test = val[0].get("match_id")
                break
            elif isinstance(val, dict) and "matches" in val:
                match_list = val["matches"]
                if len(match_list) > 0:
                    match_id_to_test = match_list[0].get("match_id")
                    break

if match_id_to_test:
    print(f"\n✅ Found matches today! We will use match_id: {match_id_to_test} for detailed tests.")
else:
    print("\n⚠️ Could not automatically extract a match_id. Using the fallback match provided.")
    match_id_to_test = 780100645 # Manually testing the match you provided earlier

## 3. Deep Dive into Stats & Events
Let's fetch the granular match data so we can map `live_xg` and `red_cards`.

In [ ]:
if match_id_to_test:
    print(f"Fetching stats for match {match_id_to_test}...")
    stats_data = fetch_footballdata(f"matches/{match_id_to_test}/stats")
    
    if stats_data and stats_data.get("success"):
        print("\n📊 STATS DATA KEYS:", list(stats_data["data"].keys()))
        if "statistics" in stats_data["data"]:
            print("\nSnippet of 'statistics':")
            print(json.dumps(stats_data["data"]["statistics"], indent=2)[:800])
        else:
            print("\nSnippet of full stats payload:")
            print(json.dumps(stats_data["data"], indent=2)[:800])
            
    print("\n" + "="*50 + "\n")
    
    print(f"Fetching events for match {match_id_to_test}...")
    events_data = fetch_footballdata(f"matches/{match_id_to_test}/events")
    
    if events_data and events_data.get("success"):
        print("\n⏱️ EVENTS DATA KEYS:", list(events_data["data"].keys()))
        if "events" in events_data["data"]:
            print("\nSnippet of 'events':")
            print(json.dumps(events_data["data"]["events"], indent=2)[:800])
        else:
            print("\nSnippet of full events payload:")
            print(json.dumps(events_data["data"], indent=2)[:800])